# 🛒 Basket Affinity Analysis — Association Rules

This notebook applies **Market Basket Analysis** using the Apriori / FP-Growth algorithm to discover product associations for cross-sell and upsell recommendations.

### Key Metrics
| Metric | Formula | Meaning |
|---|---|---|
| **Support** | freq(A∪B) / N | How often A and B appear together |
| **Confidence** | freq(A∪B) / freq(A) | P(B \| A) — if you bought A, did you buy B? |
| **Lift** | Confidence / Support(B) | How much more likely vs random chance |

## Sections
1. Setup & Data Loading
2. Build Transaction Matrix (one-hot encoded)
3. Frequent Itemsets — Apriori
4. Association Rules
5. Top Cross-Sell Recommendations
6. Visualisation

In [ ]:
import warnings
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from mlxtend.frequent_patterns import apriori, association_rules
from mlxtend.preprocessing import TransactionEncoder

warnings.filterwarnings('ignore')
plt.style.use('seaborn-v0_8-whitegrid')

MIN_SUPPORT    = 0.05  # At least 5% of baskets
MIN_CONFIDENCE = 0.30  # 30% confidence threshold
MIN_LIFT       = 1.0   # Lift > 1 means positive association

print('Libraries loaded ✓')

## 1. Load Data

In [ ]:
df = pd.read_csv('../data/sample_orders.csv', parse_dates=['order_date'])
print(f'Orders: {len(df):,} | Customers: {df["customer_id"].nunique():,}')
df.head()

## 2. Build Transaction Matrix

In [ ]:
# Group by customer: treat each customer's entire purchase history as a basket
# In production, group by order_id for true basket-level analysis

basket_data = df.groupby('customer_id')['sku'].apply(list).reset_index()
print(f'Baskets: {len(basket_data)}')
basket_data.head(10)

In [ ]:
# One-hot encode using TransactionEncoder
te = TransactionEncoder()
te_array = te.fit_transform(basket_data['sku'])
basket_matrix = pd.DataFrame(te_array, columns=te.columns_)

print(f'Transaction matrix shape: {basket_matrix.shape}')
basket_matrix.head()

## 3. Frequent Itemsets (Apriori)

In [ ]:
frequent_itemsets = apriori(
    basket_matrix,
    min_support=MIN_SUPPORT,
    use_colnames=True,
    max_len=3  # Max 3-item combinations
)

frequent_itemsets['length'] = frequent_itemsets['itemsets'].apply(len)
frequent_itemsets = frequent_itemsets.sort_values('support', ascending=False)

print(f'Frequent itemsets found: {len(frequent_itemsets)}')
frequent_itemsets.head(15)

## 4. Association Rules

In [ ]:
rules = association_rules(
    frequent_itemsets,
    metric='lift',
    min_threshold=MIN_LIFT
)

# Apply confidence filter
rules = rules[rules['confidence'] >= MIN_CONFIDENCE].copy()

# Clean up for display
rules['antecedents_str'] = rules['antecedents'].apply(lambda x: ', '.join(sorted(x)))
rules['consequents_str'] = rules['consequents'].apply(lambda x: ', '.join(sorted(x)))
rules = rules.sort_values('lift', ascending=False)

print(f'Association rules found: {len(rules)}')
rules[['antecedents_str', 'consequents_str', 'support', 'confidence', 'lift']].head(20)

## 5. Top Cross-Sell Recommendations

In [ ]:
print('\n===== TOP CROSS-SELL RULES =====')
print(f'{"IF customer buys":<25}  →  {"RECOMMEND":<20}  Confidence  Lift')
print('-' * 75)

top_rules = rules.head(10)
for _, row in top_rules.iterrows():
    print(f"{row['antecedents_str']:<25}  →  {row['consequents_str']:<20}  "
          f"{row['confidence']:.2f}        {row['lift']:.2f}x")

## 6. Visualisation

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Scatter: Support vs Confidence, bubble size = Lift
sc = axes[0].scatter(
    rules['support'],
    rules['confidence'],
    s=rules['lift'] * 50,
    c=rules['lift'],
    cmap='YlOrRd',
    alpha=0.7,
    edgecolors='gray'
)
plt.colorbar(sc, ax=axes[0], label='Lift')
axes[0].set_xlabel('Support')
axes[0].set_ylabel('Confidence')
axes[0].set_title('Support vs Confidence\n(bubble size = lift)', fontsize=13, fontweight='bold')
axes[0].axhline(y=MIN_CONFIDENCE, color='blue', linestyle='--', alpha=0.5, label=f'Min confidence ({MIN_CONFIDENCE})')
axes[0].legend()

# Heatmap of top rule pairs
if len(rules) >= 2:
    pivot_data = rules.head(10)[['antecedents_str', 'consequents_str', 'lift']]
    pivot = pivot_data.pivot_table(
        index='antecedents_str',
        columns='consequents_str',
        values='lift',
        aggfunc='max'
    ).fillna(0)

    sns.heatmap(
        pivot, ax=axes[1], cmap='Blues',
        annot=True, fmt='.2f', linewidths=0.5
    )
    axes[1].set_title('Lift Heatmap (Top Rules)', fontsize=13, fontweight='bold')
    axes[1].set_xlabel('Consequents (Recommended)')
    axes[1].set_ylabel('Antecedents (Purchased)')
    plt.setp(axes[1].get_xticklabels(), rotation=30, ha='right')
    plt.setp(axes[1].get_yticklabels(), rotation=0)
else:
    axes[1].text(0.5, 0.5, 'Not enough rules\nfor heatmap\n(increase data size)',
                 ha='center', va='center', transform=axes[1].transAxes, fontsize=12)
    axes[1].set_title('Lift Heatmap (Top Rules)', fontsize=13, fontweight='bold')

plt.tight_layout()
plt.savefig('../dashboards/screenshots/basket_affinity.png', dpi=150, bbox_inches='tight')
plt.show()

# Export
rules[['antecedents_str', 'consequents_str', 'support', 'confidence', 'lift']]\
    .round(4).to_csv('../data/association_rules.csv', index=False)
print('Rules saved → ../data/association_rules.csv ✓')